### Gold - Dimensões

Construção das dimensões finais: cliente, vendedor, produto, canal,
região e tempo. Os nomes de coluna utilizam nomenclatura de negócio em
português, dispensando tradução para uso em ferramentas de BI.

In [ ]:
%run "../00_setup/00_setup_ambiente"

In [ ]:
from pyspark.sql import functions as F

#### dim_cliente

In [ ]:
df_dim_cliente = spark.table(f"{schema_name}.silver_clientes").select(
    "customer_id", "nome_cliente",
    F.col("segmento").alias("segmento_cliente"),
    F.col("porte").alias("porte_cliente"),
    "cidade",
    "estado_uf",
    F.col("regiao_geografica").alias("regiao"),
    "data_cadastro", "status_cliente", "email", "email_valido",
)

df_dim_cliente.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{schema_name}.dim_cliente")
print("dim_cliente:", df_dim_cliente.count())

#### dim_vendedor

In [ ]:
df_silver_vendedores = spark.table(f"{schema_name}.silver_vendedores")
df_silver_canais = spark.table(f"{schema_name}.silver_canais")
df_silver_regioes = spark.table(f"{schema_name}.silver_regioes")

df_dim_vendedor = (
    df_silver_vendedores
    .join(df_silver_canais, df_silver_vendedores.canal_id == df_silver_canais.canal_id, "left")
    .join(df_silver_regioes, df_silver_vendedores.regional_code == df_silver_regioes.regiao_id, "left")
    .select(
        df_silver_vendedores.seller_id,
        df_silver_vendedores.nome_vendedor,
        df_silver_canais.nome_canal.alias("canal_venda"),
        df_silver_canais.tipo_canal,
        df_silver_regioes.regiao_nome.alias("regiao"),
        df_silver_vendedores.data_contratacao,
        df_silver_vendedores.status_vendedor,
    )
)

df_dim_vendedor.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{schema_name}.dim_vendedor")
print("dim_vendedor:", df_dim_vendedor.count())
print("  sem canal:", df_dim_vendedor.filter(F.col("canal_venda").isNull()).count())
print("  sem regiao:", df_dim_vendedor.filter(F.col("regiao").isNull()).count())

#### dim_produto

In [ ]:
df_dim_produto = spark.table(f"{schema_name}.silver_produtos").select(
    "product_id", "nome_produto", "categoria", "subcategoria",
    "familia", "tags", "preco_lista", "moeda", "status_produto",
)

df_dim_produto.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{schema_name}.dim_produto")
print("dim_produto:", df_dim_produto.count())

#### dim_canal

In [ ]:
df_dim_canal = spark.table(f"{schema_name}.silver_canais").select(
    "canal_id", "nome_canal", "tipo_canal",
    F.col("ativo_bool").alias("canal_ativo"),
)

df_dim_canal.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{schema_name}.dim_canal")
print("dim_canal:", df_dim_canal.count())

#### dim_regiao

In [ ]:
df_dim_regiao = spark.table(f"{schema_name}.silver_regioes").select(
    "regiao_id", "regiao_nome", "uf_principal", "gestor_regional",
)

df_dim_regiao.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{schema_name}.dim_regiao")
display(df_dim_regiao)

#### dim_tempo

Não existe em nenhuma fonte; gerada a partir do intervalo de order_date
do cabeçalho de pedidos, com margem de 1 ano para cada lado, cobrindo
promised_date e datas de entrega que estejam fora da janela exata dos
pedidos.

In [ ]:
from datetime import timedelta

df_cab = spark.table(f"{schema_name}.silver_pedidos_cabecalho")
datas = df_cab.select(F.min("order_date").alias("min_date"), F.max("order_date").alias("max_date")).collect()[0]

data_inicio = datas["min_date"] - timedelta(days=365)
data_fim = datas["max_date"] + timedelta(days=365)

df_dim_tempo = (
    spark.sql(f"SELECT explode(sequence(to_date('{data_inicio}'), to_date('{data_fim}'), interval 1 day)) as data")
    .withColumn("ano", F.year("data"))
    .withColumn("mes", F.month("data"))
    .withColumn("nome_mes", F.date_format("data", "MMMM"))
    .withColumn("trimestre", F.quarter("data"))
    .withColumn("dia", F.dayofmonth("data"))
    .withColumn("dia_semana", F.date_format("data", "EEEE"))
    .withColumn("ano_mes", F.date_format("data", "yyyy-MM"))
    .withColumn("ano_trimestre", F.concat(F.col("ano"), F.lit("-Q"), F.col("trimestre")))
    .withColumn("fim_de_semana", F.dayofweek("data").isin([1, 7]))
)

df_dim_tempo.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{schema_name}.dim_tempo")
print("dim_tempo:", df_dim_tempo.count(), "dias, de", data_inicio, "até", data_fim)